# SQL reference 3: aggregation and grouping

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/sql/sql-reference-03-aggregation-and-grouping.ipynb)

Aggregate functions (`count` variants, `max_by`, `listagg`, percentiles, approximate counts), `GROUP BY` rules and `HAVING`, conditional aggregation with `FILTER`, `GROUPING SETS`/`ROLLUP`/`CUBE`, `PIVOT`/`UNPIVOT`, and the classic aggregation mistakes. Practise it in [SQL practice 3: aggregation and grouping](sql-practice-03-aggregation-and-grouping.ipynb).

*Last reviewed: November 2026, against Apache Spark 4.0 (Spark SQL).*

In [ ]:
# Setup: installs the packages this notebook needs when they are missing (e.g. in Colab).
!pip install -q pyspark~=4.0.0

## Setup and sample data

A local Spark session, the shared tables as temporary views: `customers`, `orders`, `order_items`, `products`, `events` and `employees` (customer 4 has no region, customers 7 and 8 never ordered, `SKU-4014` has no product row, one event is an exact retry, two pairs of employees tie on salary, Dana out-earns her manager and Jo has no department), and three helpers: `show(query)` runs a query and prints its rows untruncated, `plan(query)` returns what `EXPLAIN` prints, and `expected_error()` prints the error class of an example that fails on purpose.

In [ ]:
import os
import sys
import tempfile
import time

from pyspark.sql import SparkSession

os.environ["PYSPARK_PYTHON"] = sys.executable  # Python workers use this notebook's interpreter
os.environ["TZ"] = "UTC"  # the sample timestamps are UTC
time.tzset()
WORK_DIR = tempfile.mkdtemp(prefix="pyspark-")  # scratch space for the files the lessons write

spark = (
    SparkSession.builder.master("local[2]")
    .config("spark.sql.shuffle.partitions", "4")  # the default of 200 is far too many for tiny data
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.warehouse.dir", os.path.join(WORK_DIR, "warehouse"))
    .config("spark.ui.enabled", "false")
    .config("spark.ui.showConsoleProgress", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

In [ ]:
# Shared sample tables as temporary views. Each statement is ordinary Spark SQL.
TABLES = {
    "customers": """CREATE OR REPLACE TEMP VIEW customers AS SELECT * FROM VALUES
  (1, 'Acme Corp',      'North', DATE'2024-01-03'),
  (2, 'Bright Retail',  'North', DATE'2024-02-11'),
  (3, 'Cyan Traders',   'South', DATE'2024-03-15'),
  (4, 'Delta Grocers',  NULL,    DATE'2024-05-02'),   -- region never arrived from the CRM
  (5, 'Echo Supplies',  'West',  DATE'2024-06-20'),
  (6, 'Foxtrot Foods',  'West',  DATE'2024-08-01'),
  (7, 'Gamma Sports',   'East',  DATE'2024-09-14'),   -- 7 and 8 have never ordered
  (8, 'Hotel Homeware', 'East',  DATE'2024-11-05')
AS customers(customer_id, name, region, signup_date)""",
    "orders": """CREATE OR REPLACE TEMP VIEW orders AS SELECT order_id, customer_id, status, order_ts, CAST(amount AS DECIMAL(12,2)) AS amount FROM VALUES
  (1,  1, 'paid',     TIMESTAMP'2026-01-05 09:15:00', 120.00),
  (2,  1, 'shipped',  TIMESTAMP'2026-01-05 10:05:00', 84.50),
  (3,  1, 'returned', TIMESTAMP'2026-01-05 11:40:00', 30.00),
  (4,  2, 'paid',     TIMESTAMP'2026-01-06 15:20:00', 260.00),
  (5,  2, 'placed',   TIMESTAMP'2026-01-07 08:05:00', 55.25),
  (6,  3, 'shipped',  TIMESTAMP'2026-01-07 17:45:00', 149.00),
  (7,  3, 'paid',     TIMESTAMP'2026-01-09 12:10:00', 310.00),
  (8,  4, 'returned', TIMESTAMP'2026-01-10 19:30:00', 89.90),
  (9,  4, 'shipped',  TIMESTAMP'2026-01-12 08:55:00', 1180.00),
  (10, 5, 'paid',     TIMESTAMP'2026-01-12 13:00:00', 74.00),
  (11, 6, 'placed',   TIMESTAMP'2026-01-13 10:25:00', 210.00),
  (12, 6, 'shipped',  TIMESTAMP'2026-01-14 00:30:00', 96.40)   -- just after midnight UTC
AS orders(order_id, customer_id, status, order_ts, amount)""",
    "order_items": """CREATE OR REPLACE TEMP VIEW order_items AS SELECT order_id, sku, qty, CAST(unit_price AS DECIMAL(8,2)) AS unit_price FROM VALUES
  (1, 'SKU-4001', 2, 12.50), (1, 'SKU-4007', 1, 95.00), (2, 'SKU-4002', 1, 34.50), (2, 'SKU-4009', 2, 25.00),
  (3, 'SKU-4002', 1, 30.00), (4, 'SKU-4003', 1, 149.00), (4, 'SKU-4004', 2, 55.50), (5, 'SKU-4005', 1, 55.25),
  (6, 'SKU-4003', 1, 149.00), (7, 'SKU-4006', 1, 189.00), (7, 'SKU-4008', 1, 121.00), (8, 'SKU-4010', 1, 89.90),
  (9, 'SKU-4011', 20, 59.00), (10, 'SKU-4012', 2, 37.00), (11, 'SKU-4013', 1, 210.00),
  (12, 'SKU-4014', 1, 41.40), (12, 'SKU-4005', 1, 55.00)
AS order_items(order_id, sku, qty, unit_price)""",
    "products": """CREATE OR REPLACE TEMP VIEW products AS SELECT sku, category, CAST(list_price AS DECIMAL(8,2)) AS list_price FROM VALUES
  ('SKU-4001', 'Kitchen', 12.50), ('SKU-4002', 'Kitchen', 34.50), ('SKU-4003', 'Garden', 149.00),
  ('SKU-4004', 'Garden', 55.50), ('SKU-4005', 'Pantry', 55.25), ('SKU-4006', 'Outdoor', 189.00),
  ('SKU-4007', 'Kitchen', 95.00), ('SKU-4008', 'Outdoor', 121.00), ('SKU-4009', 'Pantry', 25.00),
  ('SKU-4010', 'Garden', 89.90), ('SKU-4011', 'Pantry', 59.00), ('SKU-4012', 'Kitchen', 37.00),
  ('SKU-4013', 'Outdoor', 210.00)   -- SKU-4014 has no product row
AS products(sku, category, list_price)""",
    "events": """CREATE OR REPLACE TEMP VIEW events AS SELECT * FROM VALUES
  (1, 'view', TIMESTAMP'2026-01-06 09:00:00'), (1, 'add_to_cart', TIMESTAMP'2026-01-06 09:02:00'),
  (1, 'purchase', TIMESTAMP'2026-01-06 09:10:00'), (2, 'view', TIMESTAMP'2026-01-07 10:00:00'),
  (2, 'purchase', TIMESTAMP'2026-01-07 10:40:00'), (3, 'view', TIMESTAMP'2026-01-06 11:15:00'),
  (3, 'view', TIMESTAMP'2026-01-08 11:20:00'), (4, 'add_to_cart', TIMESTAMP'2026-01-09 20:05:00'),
  (4, 'purchase', TIMESTAMP'2026-01-10 08:30:00'), (5, 'view', TIMESTAMP'2026-01-11 14:45:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00'), (5, 'purchase', TIMESTAMP'2026-01-12 12:30:00'),
  (5, 'view', TIMESTAMP'2026-01-12 09:00:00')   -- an SDK retry committed this event twice
AS events(user_id, event, ts)""",
    "employees": """CREATE OR REPLACE TEMP VIEW employees AS SELECT * FROM VALUES
  (1,  'Ada',    'Engineering', CAST(NULL AS INT), 185000, DATE'2018-03-01'),
  (2,  'Brett',  'Engineering', 1,    150000, DATE'2019-07-15'),
  (3,  'Chen',   'Engineering', 1,    150000, DATE'2020-01-06'),
  (4,  'Dana',   'Engineering', 2,    155000, DATE'2022-09-12'),   -- out-earns her manager
  (5,  'Eli',    'Sales',       1,    130000, DATE'2019-02-01'),
  (6,  'Farah',  'Sales',       5,    95000,  DATE'2021-05-17'),
  (7,  'Gus',    'Sales',       5,    95000,  DATE'2023-01-09'),
  (8,  'Hana',   'Finance',     1,    140000, DATE'2017-11-20'),
  (9,  'Ivan',   'Finance',     8,    88000,  DATE'2024-04-01'),
  (10, 'Jo',     NULL,          8,    70000,  DATE'2025-06-30')   -- department not assigned yet
AS employees(emp_id, name, dept, manager_id, salary, hire_date)""",
}
for statement in TABLES.values():
    spark.sql(statement)

import contextlib
import io
import logging
import re


def show(query, n=50, **args):
    """Run a query and print up to n rows without truncating values. Keyword arguments bind :name parameters."""
    spark.sql(query, args=args or None).show(n, truncate=False)


def plan(query, mode="formatted"):
    """The plan EXPLAIN prints for a query, as text."""
    return spark.sql(f"EXPLAIN {mode.upper()} {query}").first()[0]


@contextlib.contextmanager
def expected_error():
    """Run a block that is meant to fail and print the error class instead of a stack trace."""
    spark.sparkContext.setLogLevel("OFF")
    for logger in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
        logging.getLogger(logger).setLevel(logging.CRITICAL)
    try:
        yield
    except Exception as error:
        lines = [line for line in str(error).splitlines() if line.strip()]
        message = next((line for line in lines if re.search(r"\[[A-Z][A-Z0-9_.]+\]", line)), lines[0] if lines else "")
        print(type(error).__name__ + ":", message.strip().split(". ")[0][:120])
    finally:
        spark.sparkContext.setLogLevel("ERROR")

## Contents

| section | entries |
| --- | --- |
| [Aggregate functions](#aggregate-functions) | [Core aggregates](#core-aggregates), [Picking values: `max_by`, `min_by`, `any_value`](#picking-values-max_by-min_by-any_value), [Collecting values: `collect_list`, `listagg`, `string_agg`](#collecting-values-collect_list-listagg-string_agg), [Percentiles and distribution](#percentiles-and-distribution) |
| [Grouping](#grouping) | [`GROUP BY` and `HAVING`](#group-by-and-having), [Conditional aggregation and `FILTER`](#conditional-aggregation-and-filter), [`GROUPING SETS`, `ROLLUP` and `CUBE`](#grouping-sets-rollup-and-cube) |
| [Reshaping](#reshaping) | [`PIVOT` and `UNPIVOT`](#pivot-and-unpivot), [Aggregation pitfalls](#aggregation-pitfalls) |

## Aggregate functions

### Core aggregates

| function | returns | NULLs |
| --- | --- | --- |
| `count(*)` | number of rows | counted |
| `count(expr)` / `count(DISTINCT expr)` | non-NULL values / distinct non-NULL values | ignored |
| `sum(expr)`, `avg(expr)` | total / mean | ignored; NULL when there are no non-NULL values |
| `min(expr)`, `max(expr)` | smallest / largest (numbers, strings, dates) | ignored |
| `count_if(cond)` | rows where `cond` is TRUE | |
| `bool_and(cond)` / `every`, `bool_or(cond)` / `some` / `any` | whether all / any rows satisfy `cond` | ignored |

**Notes:** `avg` of an integer column returns a `DOUBLE`; `sum` of `DECIMAL(p,s)` widens the precision so it cannot
overflow. An aggregate over zero rows (no `GROUP BY`) still returns one row: `count` 0, the rest NULL.

In [ ]:
show("""
SELECT count(*) AS orders, count(DISTINCT customer_id) AS customers, sum(amount) AS revenue,
       round(avg(amount), 2) AS avg_order, min(order_ts) AS first_order, max(amount) AS largest,
       count_if(status = 'returned') AS returns, bool_or(amount > 1000) AS any_big, bool_and(amount > 20) AS all_over_20
FROM orders
""")
show("SELECT count(*) AS n, sum(amount) AS total FROM orders WHERE status = 'cancelled'")   # one row, even with no input

### Picking values: `max_by`, `min_by`, `any_value`

| function | returns |
| --- | --- |
| `max_by(x, ord)` / `min_by(x, ord)` | the value of `x` on the row with the largest / smallest `ord` (`arg_max` in BigQuery, `max_by` in Snowflake and DuckDB) |
| `any_value(x)` / `first(x)` | an arbitrary value from the group, for columns you know are constant per group |

**Notes:** `max_by` answers "which status had the biggest order?" in one pass, without a window function or a self
join. When several rows tie on `ord`, which one wins is not defined: add a tie-breaker to `ord` (a struct or a
composite expression) if it matters.

In [ ]:
show("""
SELECT customer_id, max(amount) AS largest, max_by(order_id, amount) AS largest_order, max_by(status, amount) AS its_status,
       min_by(order_id, order_ts) AS first_order
FROM orders GROUP BY customer_id ORDER BY customer_id
""")

### Collecting values: `collect_list`, `listagg`, `string_agg`

| function | returns |
| --- | --- |
| `collect_list(x)` / `array_agg(x)` | an array of the group's non-NULL values (order not guaranteed) |
| `collect_set(x)` | an array of distinct values |
| `listagg(x, sep) WITHIN GROUP (ORDER BY ...)` / `string_agg(x, sep)` | one delimited string (Spark 4.0, Snowflake, Postgres, BigQuery) |
| `array_sort(collect_list(x))` | a deterministically ordered array |

**Notes:** `collect_list` order depends on how rows reach the aggregate; for a guaranteed order use
`listagg ... WITHIN GROUP (ORDER BY ...)` or sort the array.

In [ ]:
show("""
SELECT customer_id,
       listagg(status, ', ') WITHIN GROUP (ORDER BY order_id) AS statuses_in_order,
       array_sort(collect_set(status)) AS distinct_statuses,
       size(collect_list(order_id)) AS n
FROM orders GROUP BY customer_id ORDER BY customer_id
""")

### Percentiles and distribution

| function | returns |
| --- | --- |
| `median(x)` | the 50th percentile (interpolated) |
| `percentile_cont(p) WITHIN GROUP (ORDER BY x)` / `percentile(x, p)` | interpolated percentile (standard syntax / Spark syntax) |
| `percentile_disc(p) WITHIN GROUP (ORDER BY x)` | an actual value from the data |
| `approx_percentile(x, p [, accuracy])` | approximate, cheap on large data |
| `stddev(x)`, `variance(x)` | sample standard deviation and variance (`stddev_pop` for population) |
| `approx_count_distinct(x)` | HyperLogLog estimate, about 5% error by default, far cheaper than `count(DISTINCT)` at scale |

In [ ]:
show("""
SELECT dept, median(salary) AS median, percentile_cont(0.9) WITHIN GROUP (ORDER BY salary) AS p90,
       percentile_disc(0.5) WITHIN GROUP (ORDER BY salary) AS disc_median, round(stddev(salary)) AS sd,
       approx_count_distinct(salary) AS approx_distinct
FROM employees WHERE dept IS NOT NULL GROUP BY dept ORDER BY dept
""")

## Grouping

### `GROUP BY` and `HAVING`

```sql
SELECT keys, aggregates FROM t [WHERE row_filter] GROUP BY keys | ALL [HAVING group_filter]
```

| rule | detail |
| --- | --- |
| every non-aggregated `SELECT` expression must be a grouping key | otherwise Spark raises `MISSING_AGGREGATION` (MySQL's lenient mode silently picks a value) |
| `GROUP BY ALL` | group by every non-aggregate `SELECT` expression (Spark, Databricks, Snowflake, DuckDB) |
| `GROUP BY 1, 2` / aliases | allowed in Spark |
| NULL keys | all NULLs form **one** group |
| `WHERE` vs `HAVING` | `WHERE` filters rows before grouping (cheaper, can be pushed into the scan); `HAVING` filters groups by aggregates |

In [ ]:
show("""
SELECT customer_id, count(*) AS orders, sum(amount) AS total
FROM orders
WHERE status <> 'returned'          -- rows
GROUP BY ALL
HAVING count(*) >= 2 AND sum(amount) > 200   -- groups
ORDER BY total DESC
""")
show("SELECT dept, count(*) AS headcount FROM employees GROUP BY dept ORDER BY dept NULLS LAST")   # NULL is one group
with expected_error():
    show("SELECT dept, name, count(*) FROM employees GROUP BY dept")

### Conditional aggregation and `FILTER`

Aggregate a subset of rows per group by putting the condition inside the aggregate. Both forms below are equivalent;
`FILTER` is standard SQL (Spark, Postgres, DuckDB), `CASE` works everywhere.

```sql
sum(amount) FILTER (WHERE status = 'paid')
sum(CASE WHEN status = 'paid' THEN amount END)
```

**Use it for:** several metrics in one pass (paid vs returned revenue), manual pivots, funnel step counts. Do not
write `ELSE 0` inside `sum(CASE ...)` if you need NULL for "no rows"; do write it when you want 0.

In [ ]:
show("""
SELECT customer_id,
       sum(amount) FILTER (WHERE status = 'paid')           AS paid,
       coalesce(sum(CASE WHEN status = 'returned' THEN amount END), 0) AS returned,
       count(*) FILTER (WHERE status IN ('placed', 'paid')) AS open_or_paid
FROM orders GROUP BY customer_id ORDER BY customer_id
""")

### `GROUPING SETS`, `ROLLUP` and `CUBE`

Several groupings in one query, with subtotals:

| clause | groupings produced |
| --- | --- |
| `GROUP BY GROUPING SETS ((a, b), (a), ())` | exactly the listed sets |
| `GROUP BY ROLLUP (a, b)` | `(a, b)`, `(a)`, `()`: hierarchical subtotals and a grand total |
| `GROUP BY CUBE (a, b)` | every combination: `(a, b)`, `(a)`, `(b)`, `()` |
| `grouping(col)` | 1 when `col` is aggregated away in this row (a subtotal), else 0 |
| `grouping_id(a, b)` | a bitmask identifying the grouping set |

**Notes:** subtotal rows show NULL in the rolled-up columns, which is indistinguishable from a real NULL key unless
you check `grouping(col)`; label subtotals with `CASE WHEN grouping(col) = 1 THEN 'All' ELSE col END`.

In [ ]:
show("""
SELECT dept_label AS dept, era_label AS era, headcount, payroll
FROM (
    SELECT CASE WHEN grouping(dept) = 1 THEN 'All' ELSE coalesce(dept, 'Unassigned') END AS dept_label,
           CASE WHEN grouping(era) = 1 THEN 'All' ELSE era END AS era_label,
           count(*) AS headcount, sum(salary) AS payroll,
           grouping(dept) AS g_dept, dept AS dept_key, grouping(era) AS g_era, era AS era_key
    FROM (SELECT dept, salary, CASE WHEN hire_date < DATE'2021-01-01' THEN 'before 2021' ELSE '2021 on' END AS era FROM employees)
    GROUP BY ROLLUP (dept, era)
)
ORDER BY g_dept, dept_key NULLS LAST, g_era, era_key
""")

## Reshaping

### `PIVOT` and `UNPIVOT`

```sql
SELECT * FROM t PIVOT (agg(value) [AS alias] FOR column IN (v1 [AS name1], v2, ...))
SELECT * FROM t UNPIVOT [INCLUDE NULLS] (value_col FOR name_col IN (c1 [AS 'n1'], c2, ...))
```

| clause | does |
| --- | --- |
| `PIVOT` | turns distinct values of `column` into columns, aggregating `value`; columns not mentioned become grouping keys |
| `UNPIVOT` | turns columns into rows (`name_col`, `value_col`); NULL values are dropped unless `INCLUDE NULLS` |

**Notes:** the `IN` list must be written out: SQL cannot produce a dynamic number of columns. Select only the columns
you need before pivoting, or every extra column becomes a grouping key. Conditional aggregation (`sum(...) FILTER`) is
the portable pivot.

In [ ]:
show("""
SELECT * FROM (SELECT customer_id, status, amount FROM orders)
PIVOT (sum(amount) FOR status IN ('placed', 'paid' AS paid_total, 'shipped', 'returned'))
ORDER BY customer_id
""")
show("""
SELECT * FROM (SELECT 'Kitchen' AS category, 10 AS q1, 12 AS q2, NULL AS q3)
UNPIVOT (units FOR quarter IN (q1, q2, q3))
""")

### Aggregation pitfalls

| pitfall | fix |
| --- | --- |
| averaging averages (`avg` of per-customer `avg`) weights every customer equally | aggregate from the base rows, or weight by counts: `sum(total) / sum(n)` |
| aggregating after a one-to-many join multiplies the "one" side's amounts | aggregate each side to the join grain first, then join ([joins](sql-reference-04-joins.ipynb)) |
| `count(DISTINCT a, b)` vs `count(DISTINCT concat(a, b))` | concatenation can collide (`'ab'+'c'` = `'a'+'bc'`); use the multi-column form |
| `HAVING` without `GROUP BY` | treats the whole table as one group |

In [ ]:
show("""
SELECT round(avg(amount), 2) AS avg_order_value,                       -- every order weighs the same
       round((SELECT avg(customer_avg)
              FROM (SELECT avg(amount) AS customer_avg FROM orders GROUP BY customer_id)), 2) AS avg_of_customer_averages
FROM orders                                                             -- every customer weighs the same
""")

**Next:** practise these entries in [SQL practice 3: aggregation and grouping](sql-practice-03-aggregation-and-grouping.ipynb).